# Common table expressions

Using the northwind database

## 1. Write a CTE that lists the names and quantities of products with a unit price greater than $50.

In [ ]:
WITH HighPriceProducts AS (
    SELECT
        ProductName,
        QuantityPerUnit
    FROM
        Products
    WHERE
        UnitPrice > 50
)
SELECT
    ProductName,
    QuantityPerUnit AS Unit
FROM
    HighPriceProducts;

#### Expected results

|ProductName                     |Unit|
|--------------------------------|----|
|Mishi Kobe Niku                 |18 - 500 g pkgs.|
|Carnarvon Tigers                |16 kg pkg.|
|Sir Rodney's Marmalade          |30 gift boxes|
|Thüringer Rostbratwurst         |50 bags x 30 sausgs.|
|Côte de Blaye                   |12 - 75 cl bottles|
|Manjimup Dried Apples           |50 - 300 g pkgs.|
|Raclette Courdavault            |5 kg pkg.|


## 2. What are the top 5 most profitable products?

In [ ]:
WITH ProductRevenue AS (
    SELECT
        ProductID,
        SUM(UnitPrice * Quantity * (1 - Discount)) AS TotalRevenue
    FROM
        'Order Details'
    GROUP BY
        ProductID
),
RankedRevenue AS (
    SELECT
        p.ProductID,
        p.ProductName,
        r.TotalRevenue,
        -- Usamos RANK() o DENSE_RANK() si hay empates
        RANK() OVER (ORDER BY r.TotalRevenue DESC) as RevenueRank
    FROM
        Products AS p
    JOIN
        ProductRevenue AS r ON p.ProductID = r.ProductID
)
SELECT
    ProductID,
    ProductName,
    ROUND(TotalRevenue, 0) AS TotalRevenue
FROM
    RankedRevenue
WHERE
    RevenueRank <= 5;

#### Expected results
Alice Mutton is 6th

|ProductID                       |ProductName|TotalRevenue|
|--------------------------------|-----------|------------|
|-                             |-|-       |
|-                           |-|-       |
|-                            |-|-       |
|-                             |-|-       |
|-                              |-|-       |
|17                              |Alice Mutton|12909       |

## 3. Write a CTE that lists the top 5 categories by the number of products they have.

In [ ]:
WITH CategoryProductCount AS (
    SELECT
        CategoryID,
        COUNT(ProductID) AS ProductCount
    FROM
        Products
    GROUP BY
        CategoryID
)
SELECT
    c.CategoryName,
    t.ProductCount
FROM
    Categories AS c
JOIN
    CategoryProductCount AS t ON c.CategoryID = t.CategoryID
ORDER BY
    t.ProductCount DESC
LIMIT 5;

#### Expected result
|CategoryName                    |ProductCount|
|--------------------------------|------------|
|Confections                     |13          |
|Beverages                       |12          |
|Condiments                      |12          |
|Seafood                         |12          |
|Dairy Products                  |10          |


## 4. Write a CTE that shows the average order quantity for each product category.

WITH AvgOrderQuantityByCategory AS (
    SELECT
        p.CategoryID,
        AVG(od.Quantity) AS AvgOrderQuantity
    FROM
        "Order Details" AS od
    JOIN
        Products AS p ON od.ProductID = p.ProductID
    GROUP BY
        p.CategoryID
)
SELECT
    c.CategoryName,
    ROUND(a.AvgOrderQuantity, 4) AS AvgOrderQuantity
FROM
    Categories AS c
JOIN
    AvgOrderQuantityByCategory AS a ON c.CategoryID = a.CategoryID;

|CategoryName                    |AvgOrderQuantity|
|--------------------------------|----------------|
|Beverages                       |24.6129         |
|Condiments                      |28.2245         |
|Confections                     |25.1190         |
|Dairy Products                  |26.0100         |
|Grains/Cereals                  |21.7143         |
|Meat/Poultry                    |25.7600         |
|Produce                         |21.6667         |
|Seafood                         |21.5672         |


# 5. Create a CTE to calculate the average order amount for each customer.

In [ ]:
WITH OrderTotal AS (
    -- Calcula el total de ingresos para cada pedido
    SELECT
        OrderID,
        SUM(UnitPrice * Quantity * (1 - Discount)) AS TotalAmount
    FROM
        "Order Details"
    GROUP BY
        OrderID
),
CustomerOrderAverage AS (
    -- Une el total del pedido con la tabla Orders para obtener el CustomerID
    SELECT
        o.CustomerID,
        t.TotalAmount
    FROM
        Orders AS o
    JOIN
        OrderTotal AS t ON o.OrderID = t.OrderID
)
SELECT
    c.CustomerID,
    c.CompanyName AS CustomerName,
    ROUND(AVG(a.TotalAmount), 4) AS AvgOrderAmount
FROM
    Customers AS c
JOIN
    CustomerOrderAverage AS a ON c.CustomerID = a.CustomerID
GROUP BY
    c.CustomerID, c.CompanyName
ORDER BY
    AvgOrderAmount DESC;

|CustomerID                      |CustomerName|AvgOrderAmount|
|--------------------------------|------------|--------------|
|59                              |Piccolo und mehr|4014.2500     |
|73                              |Simons bistro|2444.3333     |
|62                              |Queen Cozinha|1991.6667     |
|51                              |Mère Paillarde|1673.8571     |
|71                              |Save-a-lot Markets|1407.2500     |
|76                              |Suprêmes délices|1345.8333     |
|81                              |Tradição Hipermercados|1315.6667     |
|7                               |Blondel père et fils|1174.4615     |
|89                              |White Clover Markets|1112.5000     |
|55                              |Old World Delicatessen|1079.5000     |
|20                              |Ernst Handel|1018.0000     |
|19                              |Eastern Connection|1004.8000     |
|68                              |Richter Supermarkt|976.6667      |
|72                              |Seven Seas Imports|934.0000      |
|63                              |QUICK-Stop  |908.5500      |
|25                              |Frankenversand|895.0667      |
|75                              |Split Rail Beer & Ale|854.7692      |
|65                              |Rattlesnake Canyon Grocery|838.2727      |
|52                              |Morgenstern Gesundkost|754.0000      |
|9                               |Bon app''   |750.4286      |
|37                              |Hungry Owl All-Night Grocers|733.5714      |
|35                              |HILARIÓN-Abastos|722.8333      |
|33                              |GROSELLA-Restaurante|690.0000      |
|34                              |Hanari Carnes|681.0000      |
|15                              |Comércio Mineiro|677.5000      |
|23                              |Folies gourmandes|672.3333      |
|10                              |Bottom-Dollar Marketse|664.0000      |
|31                              |Gourmet Lanchonetes|641.0000      |
|60                              |Princesa Isabel Vinhoss|628.5000      |
|5                               |Berglunds snabbköp|601.5556      |
|46                              |LILA-Supermercado|564.7692      |
|44                              |Lehmanns Marktstand|545.7500      |
|30                              |Godos Cocina Típica|515.0000      |
|3                               |Antonio Moreno Taquería|504.0000      |
|88                              |Wellington Importadora|503.8333      |
|47                              |LINO-Delicateses|500.0000      |
|56                              |Ottilies Käseladen|500.0000      |
|87                              |Wartian Herkku|492.5833      |
|24                              |Folk och fä HB|480.2222      |
|49                              |Magazzini Alimentari Riuniti|464.2857      |
|86                              |Die Wandernde Kuh|441.9091      |
|41                              |La maison d''Asie|437.8182      |
|8                               |Bólido Comidas preparadas|416.0000      |
|83                              |Vaffeljernet|411.0000      |
|14                              |Chop-suey Chinese|374.8333      |
|80                              |Tortuga Restaurante|357.4167      |


## 6. Sales Analysis with CTEs

Assume we have the Northwind database which contains tables like Orders, OrderDetails, and Products. Create a CTE that calculates the total sales for each product in the year 1997.

In [1]:
WITH Sales1997 AS (
    SELECT
        od.ProductID,
        SUM(od.UnitPrice * od.Quantity * (1 - od.Discount)) AS TotalSales
    FROM
        "Order Details" AS od
    JOIN
        Orders AS o ON od.OrderID = o.OrderID
    WHERE
        -- Extrae el año de la columna OrderDate
        STRFTIME('%Y', o.OrderDate) = '1997'
    GROUP BY
        od.ProductID
)
SELECT
    p.ProductName,
    ROUND(s.TotalSales, 0) AS TotalSales
FROM
    Products AS p
JOIN
    Sales1997 AS s ON p.ProductID = s.ProductID
ORDER BY
    TotalSales DESC;

SyntaxError: invalid syntax (4244347772.py, line 1)

#### Expected result

|ProductName                     |TotalSales|
|--------------------------------|----------|
|Gnocchi di nonna Alice          |173       |
|Tourtière                       |126       |
|Geitost                         |119       |
|Chang                           |115       |
|Raclette Courdavault            |115       |
|Sirop d'érable                  |106       |
|Vegie-spread                    |100       |
|Côte de Blaye                   |99        |
|Alice Mutton                    |97        |
|Steeleye Stout                  |95        |
|Sir Rodney's Scones             |92        |
|Pavlova                         |86        |
|Zaanse koeken                   |85        |
|Fløtemysost                     |75        |
|Tarte au sucre                  |75        |
